#### 04. Baselines and evaluation

This notebook mainly do the following
1. Setup checks for `config.py` (lesson 9)
2. Sanity check of `evaluate.py` on a constant forecast 
3. Log loss, AUC-PR and reliability of the constant forecast (lesson 11)
4. Threshold mapper: P(Y > θ) for any θ (lesson 12)
5. persistence-based Baseline model setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../code")          
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.functions import array_to_vector

import config as C
import evaluate as E
import mapper as M

spark = C.get_spark("task2-baselines", cores=8, memory="10g")

In [2]:
print("ROOT          ", C.ROOT)
print("paths exist   ", C.LABELS.exists(), C.ASSEMBLED.exists())
print("N_CLASSES     ", C.N_CLASSES)
print("BUCKET_SPLITS ", C.BUCKET_SPLITS)

assert C.ROOT.name == "RainfallPatternPrediction" and C.LABELS.exists() and C.ASSEMBLED.exists()
assert C.N_CLASSES == 9
assert C.BUCKET_SPLITS == [-float("inf"), 0.05, 0.25, 0.55, 1.05, 2.05, 5.05, 10.05, 20.05, float("inf")]

ROOT           /Users/lynnchen/Desktop/RainfallPatternPrediction
paths exist    True True
N_CLASSES      9
BUCKET_SPLITS  [-inf, 0.05, 0.25, 0.55, 1.05, 2.05, 5.05, 10.05, 20.05, inf]


In [3]:
TRAIN_NAME = "train_frac0.1"           
train = spark.read.parquet(str(C.ASSEMBLED / TRAIN_NAME))
val = spark.read.parquet(str(C.ASSEMBLED / "val"))

n_val, n_train = val.count(), train.count()
print(f"val {n_val:,}   {TRAIN_NAME} {n_train:,}")
assert (n_val, n_train) == (6_719_274, 5_314_338)

val 6,719,274   train_frac0.1 5,314,338


In [4]:
shares = train.groupBy("label").agg(F.sum("w").alias("sw")).toPandas().set_index("label").sw.sort_index()
shares = shares / shares.sum()
print("weighted class shares:", shares.round(5).tolist())

const = val.withColumn("probability", array_to_vector(F.array(*[F.lit(float(p)) for p in shares])))
ex = E.add_exceedance(const)
brier, rps = E.brier_rps(ex)
print(f"RPS {rps:.6f}")
# Here we print out the weighted share of each label category, for label=0, each of them 
# has w=10 while for label=1, 2, ..., each of them has w=1, then we find the share of weights
# of each category.

weighted class shares: [0.93984, 0.01907, 0.00744, 0.01071, 0.00759, 0.00726, 0.0042, 0.00281, 0.00109]


RPS 0.197678


In [5]:
# Here, the P(Y>theta) forecast is simply add up weighted weight shares of categories
# above the threshold, the val event rate is the ratio of Y>theta in samples within
# validation year 2023. 
obar = val.agg(*[F.avg((F.col("label") >= j + 1).cast("double")).alias(f"o{j}")
                 for j in range(len(C.BIN_EDGES))]).first()
P = ex.select([E.exc_col(e) for e in C.BIN_EDGES]).first()

check = pd.DataFrame([{"threshold": e,
                       "P(Y > θ) forecast": P[E.exc_col(e)],
                       "val event rate": obar[f"o{j}"],
                       "Brier (formula)": obar[f"o{j}"] * (1 - P[E.exc_col(e)]) ** 2 + (1 - obar[f"o{j}"]) * P[E.exc_col(e)] ** 2,
                       "Brier (Spark)": brier[e]}
                      for j, e in enumerate(C.BIN_EDGES)]).set_index("threshold")

assert np.allclose(check["Brier (formula)"], check["Brier (Spark)"], rtol=1e-9)
assert abs(rps - 0.197678) < 1e-6
check.round(6)

,P(Y > θ) forecast,val event rate,Brier (formula),Brier (Spark)
threshold,,,,
0.0,0.060160,0.066845,0.062422,0.062422
0.2,0.041087,0.045642,0.043580,0.043580
0.5,0.033647,0.037545,0.036151,0.036151
1.0,0.022940,0.025596,0.024948,0.024948
2.0,0.015354,0.017012,0.016725,0.016725
5.0,0.008096,0.008705,0.008630,0.008630
10.0,0.003894,0.004124,0.004107,0.004107
20.0,0.001088,0.001117,0.001116,0.001116


About Log loss, AUC-PR and reliability of the constant forecast, each new score has an exact answer for a constant forecast, which checks the code independently.
**Log loss** equals the cross-entropy between the validation class shares and the forecast shares, `−Σ ō_k log p_k`; **AUC-PR** equals the event rate at every threshold; **Reliability**: every row has the same forecast, so the table has a single bin.

In [6]:
ex = ex.cache()
ex.count()                       

ll = E.log_loss(ex)
ap = E.auc_pr(ex)                
rel = E.reliability(ex, 0.0)
ll, ap, rel

(0.36892470977192127,
 {0.0: 0.06684531691965531,
  0.2: 0.04564213336143161,
  0.5: 0.037545127643254315,
  1.0: 0.025596068861010877,
  2.0: 0.017011808120936875,
  5.0: 0.008704958303531006,
  10.0: 0.004124106265051849,
  20.0: 0.0011172337963893124},
    bin          n  mean_forecast  observed
 0    0  6719274.0        0.06016  0.066845)

In [7]:
val_shares = val.groupBy("label").count().toPandas().set_index("label")["count"].sort_index()
val_shares = val_shares / val_shares.sum()
cross_entropy = -(val_shares * np.log(shares)).sum()
print(f"log loss (MLlib) {ll:.6f}   cross-entropy formula {cross_entropy:.6f}")
assert abs(ll - cross_entropy) < 1e-8 and abs(ll - 0.368925) < 1e-6

log loss (MLlib) 0.368925   cross-entropy formula 0.368925


In [8]:
auc_check = pd.DataFrame({"threshold": C.BIN_EDGES,
                          "AUC-PR": [ap[e] for e in C.BIN_EDGES],
                          "val event rate": [obar[f"o{j}"] for j in range(len(C.BIN_EDGES))]}).set_index("threshold")
assert np.allclose(auc_check["AUC-PR"], auc_check["val event rate"], rtol=1e-6)
auc_check.round(5)

,AUC-PR,val event rate
threshold,,
0.0,0.06685,0.06685
0.2,0.04564,0.04564
0.5,0.03755,0.03755
1.0,0.02560,0.02560
2.0,0.01701,0.01701
5.0,0.00870,0.00870
10.0,0.00412,0.00412
20.0,0.00112,0.00112


The constant forecast is thus the reference for the results:

- RPS 0.1977 and log loss 0.3689.
- AUC-PR at θ = 0 is 0.0668, exactly the event rate, so it has no ranking skill.
- Reliability at θ = 0 is a single bin: forecast 0.060 against 0.067 observed, because 2023 was wetter than 2017–2022.

Every model has to beat these numbers; the baselines in the next section are the first real test. Note that Class 8 runs from 20.1 to 85.6 mm, so P(Y > θ) = 0 for θ ≥ 85.6 mm. The mapper cannot forecast a 30-minute total larger than any seen in 2017–2022; this belongs in the report's limitations.